<div style="background-color:#EAEAEA; padding:20px; border-left:5px solid #6C757D; border-radius:6px;">
<table style="width:100%; border:none;"><tr style="border:none;"><td style="border:none; vertical-align:top;">
<h1 style="font-size:32px; margin-top:0;">Master's Thesis</h1><hr style="margin:16px 0 22px 0;">
<p style="font-size:22px; line-height:1.5; margin:0;"><strong>Master's Degree in Advanced Physics</strong> - <strong>Universitat de València</strong></p>
<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:</p>
<div style="font-size:25px; font-weight:700; line-height:1.3; margin-top:14px; margin-bottom:26px;">Fast Simulation of Neutrino Oscillations in Matter</div>
<p style="font-size:14px; line-height:1.55;"><strong>Author</strong><br>Juan Ramon Diaz Santos - <a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a></p>
<p style="font-size:14px; line-height:1.55;"><strong>Supervisors</strong><br>Roberto Ruiz de Austri Bazan — <a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>Michele Lucente — <a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a></p>
<p style="font-size:14px; line-height:1.55; margin-bottom:0;"><strong>Date</strong><br>September 2026</p></td>
<td style="border:none; width:230px; padding-left:25px; text-align:right; vertical-align:top;"><img src="../../../../logo_uv.png" alt="Universitat de València" style="width:200px; margin-top:5px;"></td>
</tr></table></div>

# Bahcall Solar Neutrino Data — Online Data EDA

This notebook reads the provider's public online resources directly, without using the converted TPeanuts tables. It inventories the remote content, checks formats and units, and determines which canonical products can be generated.

Official source: [https://www.sns.ias.edu/~jnb/SNdata/sndata.html](https://www.sns.ias.edu/~jnb/SNdata/sndata.html)

## Available products

- Density: electron and MSW profiles; neutron density is derivable.
- Production: BP2000/BP2004/BS2005 radial distributions.
- Flux: total model predictions.
- Spectrum: pp, hep, 8B, CNO and 7Be line shapes.
- Probability: historical MSW benchmark tables.

## 1. Imports and remote access

In [3]:
from pathlib import Path
from io import BytesIO, StringIO
import hashlib
import json
import re
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the TPeanuts project root")


def fetch(url: str, *, timeout: int = 60) -> bytes:
    request = urllib.request.Request(url, headers={"User-Agent": "TPeanuts/solar-data"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read()


def download(url: str, path: Path, *, overwrite: bool = False) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if overwrite or not path.exists():
        path.write_bytes(fetch(url))
    return path


PROJECT_ROOT = find_project_root(Path.cwd())
SOLAR_DATA = PROJECT_ROOT / "data" / "solar"

## 2. Online inventory and format inspection

In [4]:
LANDING_URL = "https://www.sns.ias.edu/~jnb/SNdata/sndata.html"
URLS = {
    "electron_density": "https://www.sns.ias.edu/~jnb/SNdata/Export/BP2000/neordered.output",
    "msw_density": "https://www.sns.ias.edu/~jnb/SNdata/Export/BP2000/nsterile.output",
    "bp2004_production": "https://www.sns.ias.edu/~jnb/SNdata/Export/BP2004/bp2004flux.dat",
    "pp_spectrum": "https://www.sns.ias.edu/~jnb/SNdata/Export/PPenergyspectrum/ppenergytab",
    "hep_spectrum": "https://www.sns.ias.edu/~jnb/SNdata/Export/Hepspectrum/hepspectrum.dat",
    "lma_probability": "https://www.sns.ias.edu/~jnb/SNdata/Export/MSWsurvival/PNU98LMA.dat",
}
rows = []
remote = {}
for name, url in URLS.items():
    payload = fetch(url)
    remote[name] = payload.decode("utf-8", errors="replace")
    rows.append({"resource": name, "url": url, "bytes": len(payload), "sha256": hashlib.sha256(payload).hexdigest()})
display(pd.DataFrame(rows))
print(remote["electron_density"][:500])

,resource,url,bytes,sha256
0,electron_density,https://www.sns.ias.edu/~jnb/SNdata/Export/BP2...,60942,32edea4daa671e115ab2f79a40c290bc5cf68c7400b107...
1,msw_density,https://www.sns.ias.edu/~jnb/SNdata/Export/BP2...,87748,f51f6b384ff5a6ae7f7c7ff232b6690c555871f07959b4...
2,bp2004_production,https://www.sns.ias.edu/~jnb/SNdata/Export/BP2...,78214,49c3e34a8521c5b0292f83ab7eef66332615b729d81a95...
3,pp_spectrum,https://www.sns.ias.edu/~jnb/SNdata/Export/PPe...,1476,f350a6ad075a3e06fdc6990aa4de687b79c2af954ec807...
4,hep_spectrum,https://www.sns.ias.edu/~jnb/SNdata/Export/Hep...,26000,792b9e5b61e133cc136f8cde38f46dba7e749f8cf0c693...
5,lma_probability,https://www.sns.ias.edu/~jnb/SNdata/Export/MSW...,13901,732430bf8f2e3a4078ed4930ac273f9bcc02f3467c156c...


     BP2000 Electron Density
         astro-ph/001034

     8.19005E-03    2.00852
     8.31693E-03    2.00847
     8.44579E-03    2.00842
     8.57666E-03    2.00837
     8.70957E-03    2.00832
     8.84456E-03    2.00826
     8.98165E-03    2.00820
     9.12087E-03    2.00814
     9.26227E-03    2.00808
     9.40589E-03    2.00802
     9.55174E-03    2.00795
     9.69987E-03    2.00788
     9.85031E-03    2.00781
     1.00031E-02    2.00774
     1.01583E-02    2.00767
     1.03159E-02    2.007


## 3. Conclusions

Only source-published observables are attributed to this provider. Derived TPeanuts quantities and detector-level spectra are labelled separately by the generator notebook.